In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)

from sklearn.metrics import precision_recall_fscore_support, roc_auc_score


In [2]:
train_data_path = 'DataFolder/train_masked.csv'
val_data_path   = 'DataFolder/val_masked.csv'
df_train = pd.read_csv(train_data_path)
df_val  = pd.read_csv(val_data_path)
df_train.head()

,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation
0,1109,Well first you have to figure out if there is ...,No legal advice: Do not offer or request legal...,relationships,Perhaps offer him a plea deal of 20 minutes of...,"Firstly, the act of stealing the emails was il...",*that is more or less a binding agreement*\n\n...,But did he yell surprise? It's not rape if you...,1
1,490,cheap cigarettes online\nBuy Discount Duty Fre...,"No Advertising: Spam, referral links, unsolici...",pics,'m giving out Tyrande codes for 4 dollars upfr...,If anybody wants free case to open. Here is pr...,cauperheinea1970.tumblr.com - cam 2 cam now ch...,ヽ༼ ຈل͜ຈ༽ ﾉ Raise Them!\n\n ^^Dongers ^^Raised:...,1
2,159,"It's illegal, you can sue for pain and sufferi...",No legal advice: Do not offer or request legal...,legaladvice,Get a lawyer and get the security camera foota...,"Dear dumbass, they stole $1700 dollars from hi...",Can you beat and rape her? Then get her pregna...,Depends how much you want to keep your liquor ...,1
3,333,You should be fine. There have been cases wher...,No legal advice: Do not offer or request legal...,relationships,State dependant. There are states where it's p...,"make multiple deposits to multiple banks, stay...","Wait a few month, get her to some stairs and l...",But it's cheaper to just shoot them than to pe...,1
4,292,[Also Watch This Video - Olympics]([URL_dynami...,"No Advertising: Spam, referral links, unsolici...",videos,I just found that you can get 100 000 Pokemon ...,hunt for lady for jack off in neighbourhood [U...,SD Stream [English Stream]([URL_mntvlive]),"\nHey, you do not want to cmprar my awp asiimo...",0


In [3]:


class PairRedditRulesDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=384):
        self.labels = df["rule_violation"].astype(int).tolist()
        self.comment = df["body"].astype(str).tolist()
        self.context = (
            "RULE: " + df["rule"].astype(str) + "\n"
            "SUBREDDIT: " + df["subreddit"].astype(str) + "\n"
            "POS: " + df["positive_example_1"].fillna("").astype(str) + " || " +
                     df["positive_example_2"].fillna("").astype(str) + "\n"
            "NEG: " + df["negative_example_1"].fillna("").astype(str) + " || " +
                     df["negative_example_2"].fillna("").astype(str)
        ).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, i):
        enc = self.tokenizer(
            self.comment[i],
            self.context[i],
            truncation="longest_first",   # <— change this
            max_length=self.max_len,
            padding=False,
            return_tensors=None
        )
        enc["labels"] = int(self.labels[i])
        return enc


In [4]:

model_name = "roberta-base"
#model_name = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast = True,  model_max_length=512)

train_pairds = PairRedditRulesDataset(df_train, tokenizer, max_len=512)
val_pairds = PairRedditRulesDataset(df_val, tokenizer, max_len=512)
collate = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

train_dataloader = DataLoader(train_pairds, batch_size=32, 
                                shuffle=True,
                                collate_fn=collate)

val_dataloader = DataLoader(val_pairds, batch_size=32,
                            shuffle=False, 
                            collate_fn=collate)

batch = next(iter(train_dataloader))

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    probs_pos = (np.exp(logits) / np.exp(logits).sum(-1, keepdims=True))[:, 1]
    try:
        auroc = roc_auc_score(labels, probs_pos)
    except Exception:
        auroc = float("nan")
    acc = (preds == labels).mean()
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "auroc": auroc}


In [7]:
import transformers, tokenizers, sys, torch
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("python:", sys.version)
print("torch:", torch.__version__)
print("transformers path:", transformers.__file__)


transformers: 4.57.1
tokenizers: 0.22.1
python: 3.11.13 | packaged by conda-forge | (main, Jun  4 2025, 14:52:34) [Clang 18.1.8 ]
torch: 2.3.1
transformers path: /opt/anaconda3/envs/dml/lib/python3.11/site-packages/transformers/__init__.py


In [ ]:

for p in model.base_model.parameters():
    p.requires_grad = False
for p in model.classifier.parameters():
    p.requires_grad = True

args_head = TrainingArguments(
    output_dir="./rb-rule-clf",
    per_gpu_train_batch_size=16,     
    per_gpu_eval_batch_size=32,     
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    learning_rate=1e-4,
    warmup_steps=200,                 
    weight_decay=0.01,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args_head,
    train_dataset=train_pairds,
    eval_dataset=val_pairds,
    tokenizer=tokenizer,
    data_collator=collate,
    compute_metrics=compute_metrics,
)

trainer.train()


/var/folders/vv/m7pn5lyj3nsbw8_qt0cs62tr0000gn/T/ipykernel_12900/2654308428.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Using deprecated `--per_gpu_train_batch_size` argument which will be removed in a future version. Using `--per_device_train_batch_size` is preferred.
Using deprecated `--per_gpu_train_batch_size` argument which will be removed in a future version. Using `--per_device_train_batch_size` is preferred.


Step,Training Loss
50,0.695800


TrainOutput(global_step=51, training_loss=0.6954718828201294, metrics={'train_runtime': 190.742, 'train_samples_per_second': 8.509, 'train_steps_per_second': 0.267, 'total_flos': 312247044948480.0, 'train_loss': 0.6954718828201294, 'epoch': 1.0})

In [9]:
from torch.optim import AdamW

encoder_layers = model.base_model.encoder.layer
for layer in encoder_layers[-2:]:
    for p in layer.parameters():
        p.requires_grad = True

encoder_params = []
for layer in encoder_layers[-2:]:
    encoder_params += list(layer.parameters())
head_params = list(model.classifier.parameters())

optimizer = AdamW([
    {"params": encoder_params, "lr": 1.5e-5},
    {"params": head_params,    "lr": 8e-5},
])

args_ft = TrainingArguments(
    output_dir="./rb-rule-clf-ft",
    per_gpu_train_batch_size=16,
    per_gpu_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=1,         
    learning_rate=2e-5,         
    warmup_steps=200,
    weight_decay=0.01,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args_ft,
    train_dataset=train_pairds,
    eval_dataset=val_pairds,
    tokenizer=tokenizer,
    data_collator=collate,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),   
)

trainer.train()


/var/folders/vv/m7pn5lyj3nsbw8_qt0cs62tr0000gn/T/ipykernel_12900/3680349368.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Using deprecated `--per_gpu_train_batch_size` argument which will be removed in a future version. Using `--per_device_train_batch_size` is preferred.
Using deprecated `--per_gpu_train_batch_size` argument which will be removed in a future version. Using `--per_device_train_batch_size` is preferred.


Step,Training Loss
50,0.692600


TrainOutput(global_step=51, training_loss=0.6922408248863968, metrics={'train_runtime': 412.4739, 'train_samples_per_second': 3.935, 'train_steps_per_second': 0.124, 'total_flos': 312247044948480.0, 'train_loss': 0.6922408248863968, 'epoch': 1.0})

In [10]:
val_metrics = trainer.evaluate()
print(val_metrics)


Using deprecated `--per_gpu_eval_batch_size` argument which will be removed in a future version. Using `--per_device_eval_batch_size` is preferred.
Using deprecated `--per_gpu_eval_batch_size` argument which will be removed in a future version. Using `--per_device_eval_batch_size` is preferred.


Using deprecated `--per_gpu_eval_batch_size` argument which will be removed in a future version. Using `--per_device_eval_batch_size` is preferred.


{'eval_loss': 0.6905367374420166, 'eval_accuracy': 0.5246305418719212, 'eval_precision': 0.5619047619047619, 'eval_recall': 0.28640776699029125, 'eval_f1': 0.37942122186495175, 'eval_auroc': 0.5954611650485437, 'eval_runtime': 27.9605, 'eval_samples_per_second': 14.52, 'eval_steps_per_second': 0.465, 'epoch': 1.0}
